Copyright 2021 The TensorFlow Authors.

#Assess privacy risks with TensorFlow Privacy Secret Sharer Attack

In this colab, we demonstrate how [secret sharer](https://arxiv.org/abs/1802.08232) works on a language model. We will train a model with "secrets", i.e. random sequences, inserted in the training data, and then evaluate if the model has "memorized" those secrets.

(The code is largely based on a Tensorflow tutorial for training a character-level LSTM on Shakespeare dataset.)

In [ ]:

%pip install tf-models-official
#!pip install git+https://github.com/tensorflow/privacy.git
#clear_output()

In [ ]:
import functools
import os
import numpy as np
import tensorflow as tf


from tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets import SecretConfig, TextSecretProperties, generate_text_secrets_and_references, construct_secret
from tensorflow_privacy.privacy.privacy_tests.secret_sharer.exposures import compute_exposure_interpolation, compute_exposure_extrapolation

# Define functions

In [ ]:


SEQ_LENGTH = 1024

TRAIN_EPOCHS = 2
BATCH_SIZE = 32
DATASET_SIZE = 1400 #20000




In [ ]:
pip install pandas

In [ ]:
import pandas as pd
import csv
def load_text_from_csv(path_to_file):
    df = pd.read_csv(path_to_file, usecols=["TEXT"], dtype=str, encoding="ISO-8859-1", quoting=csv.QUOTE_NONE) 
    text = " ".join(df["TEXT"].dropna().iloc[:DATASET_SIZE*2].tolist()) 
    return text

In [ ]:
#@title Function for getting dataset combined with secrets.
def get_dataset(secrets=None,
                batch_size=None, seq_length=SEQ_LENGTH):
  """Creates a dataset from a given text file.

  Args:
    secrets: Secrets to be inserted.
    batch_size: Batch size to use.
    seq_length: The length of the LSTM sequence.

  Returns:
    A tuple, consisting of the Dataset and the class to character mapping
    and character to class mapping.
  """
  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  text = load_text_from_csv("NOTEEVENTS_deanonymized.csv")


  def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text


  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.encode(text, return_tensors="tf")[0]
  seq_length = 1024  # GPT-2 uses max sequence of 1024 tokens
  dataset = tf.data.Dataset.from_tensor_slices(input_ids)
  dataset = dataset.batch(seq_length + 1, drop_remainder=True)
  def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

  dataset = dataset.map(split_input_target)
  dataset = dataset.shuffle(1000)
    
  train_dataset = dataset.take(DATASET_SIZE).batch(8, drop_remainder=True)
  val_dataset = dataset.skip(DATASET_SIZE).batch(8, drop_remainder=True)
  return train_dataset, val_dataset

In [ ]:
import os
import tensorflow as tf
from transformers import TFAutoModelForCausalLM, create_optimizer, AutoTokenizer, AdamWeightDecay
from tensorflow.keras.optimizers import SGD
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer
from tensorflow.keras.callbacks import History

def train_model(dataset, validation_dataset, batch_size, vocab_size, train_epochs, checkpoint_dir, dp):
    """Trains Transformer model with checkpoints."""

    train_steps = DATASET_SIZE // batch_size 
    print("train steps", train_steps)
    model = TFAutoModelForCausalLM.from_pretrained("gpt2")
    noise_multiplier = 0.4272711315437153
    l2_norm_clip = 1e-6
    num_microbatches = BATCH_SIZE
    if dp:
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("Using DP-SGD with noise_multiplier =", noise_multiplier)
    else:
        optimizer = SGD(learning_rate=0.03)

    model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), jit_compile=True)


    callbacks = []

    if checkpoint_dir:
        checkpoint_filepath = os.path.join(checkpoint_dir, 'ckpt_{epoch}')
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_filepath,
            save_weights_only=True,
            save_best_only=True,  # Saves best model based on loss
            monitor="loss",  
            mode="min",
            verbose=1
        )
        callbacks.append(checkpoint_callback)


        checkpoint_latest = tf.train.latest_checkpoint(checkpoint_dir)
        if checkpoint_latest:
            model.load_weights(checkpoint_latest)
            initial_epoch = int(os.path.basename(checkpoint_latest).split('_')[-1])
            print(f'Checkpoint found: {checkpoint_latest}. Resuming from epoch {initial_epoch}.')
        else:
            initial_epoch = 0
    else:
        initial_epoch = 0


    history = model.fit(
        dataset,
        epochs=train_epochs,
        validation_data=validation_dataset,
        initial_epoch=initial_epoch,
        steps_per_epoch=train_steps,
         callbacks=[History()],
        verbose=2
    )

    return history, callbacks

In [ ]:
from transformers import TFAutoModelForCausalLM, AutoModelForCausalLM
import tensorflow as tf


def load_gpt2_model(model_checkpoint, checkpoint_dir, dp, pytorch=False):
    """Loads GPT-2 with weights from previous checkpoint."""



    prediction_model = TFAutoModelForCausalLM.from_pretrained(model_checkpoint)
    
    if dp:
        noise_multiplier = 0.4272711315437153   #DP for Epsilon 10, delta= dataset_size^-1
        l2_norm_clip = 1e-6
        num_microbatches = BATCH_SIZE
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=0.03)
    

    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        prediction_model.load_weights(latest_checkpoint)
        print(f'Pesos cargados desde {latest_checkpoint}')
    else:
        print('No se encontró ningún checkpoint, cargando pesos preentrenados.')


    prediction_model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

    return prediction_model


In [ ]:
pip install Faker

In [ ]:
from faker import Faker
import random

fake = Faker()


def fake_name() -> str:
    return fake.last_name()

def fake_firstName() -> str:
    return fake.first_name()

def fake_hospital() -> str:
    return fake.company() + " Hospital"

def fake_hospitalUnit() -> str:
    return random.choice(["Oncology unit", "Critical Care unit", "Psychiatry unit", "Pulmonary unit", "Cardiology unit", "Ginecology unit"])

def fake_date() -> str:
    return str(fake.date_this_decade())

def fake_location() -> str:
    return str(fake.street_address())

def fake_number() -> str:
    return str(random.randint(1, 1000))

def fake_phone() -> str:
    return str(fake.phone_number())

def fake_month() -> str:
    return fake.month_name()

def fake_country() -> str:
    return fake.country()

def fake_state() -> str:
    return fake.state()

def fake_year() -> str:
    return str(random.randint(2000, 2022))

def fake_over90_age() -> str:
    return str(random.randint(90, 110))

In [186]:
import csv
from tqdm import tqdm
from itertools import islice
import json
import re
import numpy as np

def read_secrets(input_csv, amount_secrets, maxRepetitions=20, minRepetitions=10):
    secrets = []
    current_data = "" 
    
    with open(input_csv, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)  
        for i, row in enumerate(tqdm(reader)):
            if amount_secrets != -1 and i >= amount_secrets:
                break
            repetitions = row['repetitions']
            if (repetitions != "repetitions" and int(repetitions)<maxRepetitions and int(repetitions)>minRepetitions):
                phrase = row['phrase']
                secretType = row['type']
                secrets.append({
                    'repetitions': int(repetitions), 
                    'phrase': phrase,
                    'type':secretType
                })

    return secrets


def generate_reference_secret(phrase, secretType, amountReferences):
    splittedPhrase = phrase.split()
    subphrase = " ".join(splittedPhrase[:-1])
    secret = splittedPhrase[-1] 
    print("SECRET FOUND:", secret)
    ret = []
    secretWords = [secret]
    match secretType:
        case "doctorData":
            for i in range(amountReferences):
                fakedWord = fake_name()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "hospitalData":
            for i in range(amountReferences):
                fakedWord = fake_hospital()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "hospitalUnitData":
            for i in range(amountReferences):
                fakedWord = fake_hospitalUnit()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "dateData":
            for i in range(amountReferences):
                fakedWord = fake_date()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "locationData":
            for i in range(amountReferences):
                fakedWord = fake_location()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "firstNameData":
            for i in range(amountReferences):
                fakedWord = fake_firstName()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "phoneData":
            for i in range(amountReferences):
                fakedWord = fake_phone()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "monthData":
            for i in range(amountReferences):
                fakedWord = fake_month()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "countryData":
            for i in range(amountReferences):
                fakedWord = fake_country()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "genericNumberData":
            for i in range(amountReferences):
                fakedWord = fake_number()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "yearData":
            for i in range(amountReferences):
                fakedWord = fake_year()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "ageData":
            for i in range(amountReferences):
                fakedWord = fake_over90_age()
                fakedSecret = " " + fakedWord
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
                secretWords.append(fakedWord)
        case "other":
            print("Tipo de secreto no encontrado")

    return ret, secretWords
     
    
numbers= ["0","1","2","3","4","5","6","7","8","9"]
    
def generate_text_secrets_and_references(input_csv, amount_secrets = -1, amountReferences=10, maxRepetitions=50, minRepetitions=1):
    
    secrets = read_secrets(input_csv=input_csv, amount_secrets=amount_secrets, maxRepetitions=maxRepetitions, minRepetitions= minRepetitions)
    print(len(secrets),"secrets loaded")
    allowed_vocab = ["name", "hospital", "date", "address", "phone", "month", "country", "state", "year", "age", "<|endoftext|>"] + numbers
    all_reference_secrets = []
    for secret in tqdm(secrets):
      secret["datasets"] = {}
      r = secret["repetitions"]
      seqs = secret["phrase"]
      sType = secret["type"]

      indexed_seqs = np.array([tokenizer.encode(seqs, padding='max_length', max_length=SEQ_LENGTH)])
      secret["datasets"][r] = {
        'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
        #'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
      }
      reference_secrets, fakedWords = generate_reference_secret(phrase=seqs, secretType = sType, amountReferences=amountReferences)
      allowed_vocab = allowed_vocab + fakedWords #Used for model guiding while calculating perplexities
      for reference_secret in reference_secrets:
            indexed_seqs = np.array([tokenizer.encode(reference_secret, padding='max_length', max_length=SEQ_LENGTH)])
            data = {
            'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
            'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
            }
            transformed_reference_secret = {}
            transformed_reference_secret["datasets"] = {}
            transformed_reference_secret["datasets"][0] = data
        
            all_reference_secrets.append(transformed_reference_secret)
    secrets = np.concatenate((secrets, all_reference_secrets))
   
    return secrets,allowed_vocab





# Calculating perplexity on GPT2 finetuned with MIMIC-III

In [240]:
from tensorflow.nn import softmax
import numpy as np
import tensorflow as tf
from tqdm import tqdm

def compute_perplexity_for_secret(secrets, checkpoint_dir, vocab_size, allowedVocab, dp=False):
    """Get perplexity for given examples.

    Args:
        secrets: a list, where each element is a secret dictionary
        checkpoint_dir: the directory from which to load checkpoints
        vocab_size: vocabulary size
        allowedVocab: Array of possible word that the model can output (masks the predicted distribution)
        dp: If set to True, load_gpt2_model loads DP-SGD optimizer instead of SGD.
    Returns:
        Perplexity for the input examples.
    """
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

    cce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
    ppls = [{}]
    ppls_reference = [None]
    allowedTokens = []
    for vocabWord in allowedVocab:
        tokenized = tokenizer.encode(vocabWord, add_special_tokens=False)  # Get tokenized IDs
        allowedTokens.extend(tokenized) 
        
    allowedTokens = list(set(allowedTokens))
    for i, secret in enumerate(secrets):
        print("i:", i)
        for r in secret["datasets"]:
            d = secret["datasets"][r]
            indexed_seqs = d['data'] 
            all_losses = []
    
            for seq_batch in indexed_seqs:
                for seq in seq_batch:
                    tokens = tf.convert_to_tensor([seq]) 
                    allowedTokens = list(set(allowedTokens + list(seq.numpy())))
                    logits = prediction_model.predict(tokens).logits
                    mask = np.full((1, logits.shape[1], vocab_size), -np.inf)  
                    mask[:, :, allowedTokens] = 0 
                    filtered_logits = logits + mask
                    predictions_prob = softmax(filtered_logits, axis=-1)

                    loss = 0
                    realLength = 1 #Seq has padding, repeats 2056 after the phrase ends, with RealLength we count tokens to get real phrase length
                    for k in range(1, len(seq)): 
                        token = seq[k]  # Current token
                        prob = predictions_prob[0, k-1, token].numpy()  # Probability for token i given previous tokens
                        loss -= np.log2(prob)  
                        realLength += 1
                        #print("Prob of token", tokenizer.decode(token.numpy()), "is", prob)
                        if (token.numpy() == 50256):
                            break


                    all_losses.append(loss / realLength)  # Average loss over the sequence length


        print("LOSS:", all_losses)
        if r == 0:
            ppls_reference[0] = all_losses
        else:
            ppls[0][r] = all_losses

    return ppls, ppls_reference


# Run training and secret sharer on MIMIC-III dataset.

In [252]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


train_dataset, validation_dataset = get_dataset(
    batch_size=BATCH_SIZE)
    
checkpoint_dir = 'checkpoints'
print("dataset", "loaded")
history, callbacks = train_model(train_dataset, validation_dataset, batch_size=BATCH_SIZE, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=False)

Token indices sequence length is longer than the specified maximum sequence length for this model (31725 > 1024). Running this sequence through the model will result in indexing errors


dataset loaded
train steps 43


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Epoch 1/2
Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


I0000 00:00:1743473547.106275     821 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


ValueError: Expected input data to be non-empty.

In [245]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets, allowedVocab = generate_text_secrets_and_references(input_csv="secrets.txt", amount_secrets=100, amountReferences=10, maxRepetitions = 20, minRepetitions = 1)

100it [00:00, 350987.78it/s]


29 secrets loaded


  7%|▋         | 2/29 [00:00<00:01, 14.40it/s]

SECRET FOUND: 2023-11-22
SECRET FOUND: 2021-05-16
SECRET FOUND: 2021-02-28


 21%|██        | 6/29 [00:00<00:01, 14.61it/s]

...

 86%|████████▌ | 25/29 [00:01<00:00, 13.82it/s]

SECRET FOUND: 894
SECRET FOUND: 2021-10-30
SECRET FOUND: 2023-09-05


 93%|█████████▎| 27/29 [00:02<00:00, 12.82it/s]

SECRET FOUND: 2020-04-08
SECRET FOUND: Cline
SECRET FOUND: 2020-11-06


100%|██████████| 29/29 [00:02<00:00, 13.15it/s]


In [246]:
print(len(secrets))
print(len(allowedVocab))

309
330


In [247]:

ppls_all, ppls_reference_all = [], []

checkpoint_dir =  'checkpoints'
ppls, ppls_reference = compute_perplexity_for_secret(
  secrets, checkpoint_dir, tokenizer.vocab_size, allowedVocab)
ppls_all += ppls
ppls_reference_all += ppls_reference




exposure_interpolation = compute_exposure_interpolation(ppls_all[0], ppls_reference_all[0])


print('  Interpolation:', '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


No se encontró ningún checkpoint, cargando pesos preentrenados.
i: 0


  0%|          | 0/1 [00:00<?, ?it/s]

1/1 [==============================] - 2s 2s/step


100%|██████████| 1/1 [00:03<00:00,  3.30s/it]


...

LOSS: [3.4444380545240043]
i: 308


  0%|          | 0/1 [00:00<?, ?it/s]

1/1 [==============================] - 1s 622ms/step


100%|██████████| 1/1 [00:01<00:00,  1.43s/it]

LOSS: [3.5344242501174317]
  Interpolation: avg_exposure=-1.00±0.00
